# TP3 — Limpieza de datos (3.2)

**Investigación de Operaciones II (11.87) — RedBaires S.A.**

Este notebook ejecuta las 7 etapas de limpieza definidas en la sección 3.2 del informe, sobre `muestra_cascadas_REDBAIRES.csv`. Cada paso se aplica y se verifica antes de pasar al siguiente. El resultado es `muestra_cascadas_REDBAIRES_limpio.csv`, el dataset que alimenta el ajuste de distribuciones de 3.3.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

RUTA_CRUDO = '../Datos/muestra_cascadas_REDBAIRES.csv'
RUTA_LIMPIO = '../Datos/muestra_cascadas_REDBAIRES_limpio.csv'

df = pd.read_csv(RUTA_CRUDO)
df['forward_datetime'] = pd.to_datetime(df['forward_datetime'])
print(f"Filas: {len(df):,} | Columnas: {df.shape[1]} | Cascadas: {df['cascade_id'].nunique():,}")


Filas: 66,153 | Columnas: 30 | Cascadas: 5,000


## Diagnóstico inicial

Estado del dataset crudo, antes de cualquier limpieza — referencia para comparar contra cada paso.

In [2]:
print("Duplicados de event_id:", df['event_id'].duplicated().sum())
print("\n% de faltantes por columna (solo columnas con faltantes):")
faltantes = df.isna().mean().sort_values(ascending=False)
print((faltantes[faltantes > 0] * 100).round(2))


Duplicados de event_id: 132

% de faltantes por columna (solo columnas con faltantes):
parent_event_id            7.57
dwell_time_sec             3.04
sentiment_score            2.04
device_fingerprint_code    2.01
comments                   1.42
content_half_life_min      1.01
impressions                0.94
dtype: float64


## Paso 1 — Deduplicar `event_id`

El export crudo trae 132 filas con `event_id` repetido. Nos quedamos con la primera ocurrencia de cada una.

In [3]:
n_antes = len(df)
df = df.drop_duplicates(subset='event_id', keep='first').reset_index(drop=True)
n_despues = len(df)

print(f"Filas eliminadas: {n_antes - n_despues}")
assert df['event_id'].duplicated().sum() == 0, "Todavia quedan event_id duplicados"
print("OK: no quedan event_id duplicados.")


Filas eliminadas: 132
OK: no quedan event_id duplicados.


## Paso 2 — Normalizar mayúsculas/minúsculas

`content_type_code`, `platform_surface_code` y `language_code` tienen el mismo valor escrito en mayúscula y en minúscula (ej. `IMG-MACRO` / `img-macro`). Se normaliza todo a minúscula.

In [4]:
cols_case = ['content_type_code', 'platform_surface_code', 'language_code']

print("Valores únicos ANTES de normalizar:")
for c in cols_case:
    print(f"  {c}: {df[c].nunique()}")

for c in cols_case:
    df[c] = df[c].str.lower()

print("\nValores únicos DESPUÉS de normalizar:")
for c in cols_case:
    print(f"  {c}: {df[c].nunique()}")
    assert df[c].nunique() <= 6, f"{c} sigue teniendo mas variantes de las esperadas"
print("\nOK: normalizado.")


Valores únicos ANTES de normalizar:
  content_type_code: 12
  platform_surface_code: 10
  language_code: 10

Valores únicos DESPUÉS de normalizar:
  content_type_code: 6
  platform_surface_code: 5
  language_code: 5

OK: normalizado.


## Paso 3 — Truncar valores imposibles

`dwell_time_sec` e `inter_event_time_min` no pueden ser negativos (son tiempos). Se inspecciona cuántas filas están afectadas y se truncan a 0 (son magnitudes chicas, consistentes con error de redondeo de reloj, no con corrupción grave de datos).

In [5]:
for c in ['dwell_time_sec', 'inter_event_time_min']:
    neg = df[c] < 0
    print(f"{c}: {neg.sum()} filas negativas | min={df[c].min():.2f} | peor caso entre las negativas={df.loc[neg, c].min():.2f}")

df['dwell_time_sec'] = df['dwell_time_sec'].clip(lower=0)
df['inter_event_time_min'] = df['inter_event_time_min'].clip(lower=0)

assert (df['dwell_time_sec'].dropna() >= 0).all()
assert (df['inter_event_time_min'].dropna() >= 0).all()
print("\nOK: truncado a 0, no quedan valores negativos.")


dwell_time_sec: 66 filas negativas | min=-49.82 | peor caso entre las negativas=-49.82
inter_event_time_min: 39 filas negativas | min=-9.30 | peor caso entre las negativas=-9.30

OK: truncado a 0, no quedan valores negativos.


## Paso 4 — Verificar que cada cascada tiene exactamente 1 semilla

En el diagnóstico inicial (sobre los datos crudos) se habían detectado 10 cascadas con `seed_flag=1` en dos eventos. Al investigarlas, resultaron ser un efecto colateral de los 132 `event_id` duplicados del Paso 1: la fila semilla estaba duplicada, y eso inflaba el conteo a 2. No es una anomalía estructural aparte — se verifica acá que deduplicar ya la resolvió, sin necesidad de excluir ninguna cascada.

In [6]:
semillas_por_cascada = df.groupby('cascade_id')['seed_flag'].sum()
cascadas_problema = semillas_por_cascada[semillas_por_cascada != 1].index.tolist()

print(f"Cascadas con semilla != 1 (despues de deduplicar en el Paso 1): {len(cascadas_problema)}")

assert len(cascadas_problema) == 0, "Quedan cascadas sin exactamente 1 semilla; revisar antes de continuar"
assert (semillas_por_cascada == 1).all()
print(f"OK: {df['cascade_id'].nunique()} cascadas, todas con exactamente 1 semilla. No se excluyo ninguna.")


Cascadas con semilla != 1 (despues de deduplicar en el Paso 1): 0
OK: 5000 cascadas, todas con exactamente 1 semilla. No se excluyo ninguna.


## Paso 5 — Tratar faltantes

Todas las columnas con faltantes están por debajo del 3%. Se imputa:
- **`content_half_life_min`**: es constante por `message_id` (ver 3.1) — se completa primero con el valor del propio mensaje si existe en otra fila, y recién si falta en todo el mensaje, con la mediana global.
- **`impressions`, `comments`, `dwell_time_sec`, `sentiment_score`**: mediana de la columna.
- **`device_fingerprint_code`**: es un identificador, no una magnitud — no tiene sentido imputar con la moda. Se completa con la categoría `"UNKNOWN"`.

In [7]:
# parent_event_id queda afuera: su NaN es estructural (las 5.000 semillas no tienen padre), no un faltante a imputar
cols_a_tratar = [c for c in df.columns if c != 'parent_event_id']

print("Faltantes antes de imputar (sin contar parent_event_id):")
print((df[cols_a_tratar].isna().sum()[df[cols_a_tratar].isna().sum() > 0]))

# content_half_life_min: primero completar dentro del mismo mensaje
# (si TODO el mensaje esta vacio, s.median() de ese grupo da NaN con un RuntimeWarning esperado;
#  se silencia porque el fallback a la mediana global, abajo, ya lo cubre)
import warnings
with warnings.catch_warnings():
    warnings.filterwarnings('ignore', category=RuntimeWarning)
    df['content_half_life_min'] = df.groupby('message_id')['content_half_life_min'] \
        .transform(lambda s: s.fillna(s.median()))
# si todo el mensaje estaba vacio, cae a la mediana global
df['content_half_life_min'] = df['content_half_life_min'].fillna(df['content_half_life_min'].median())

for c in ['impressions', 'comments', 'dwell_time_sec', 'sentiment_score']:
    df[c] = df[c].fillna(df[c].median())

df['device_fingerprint_code'] = df['device_fingerprint_code'].fillna('UNKNOWN')

print("\nFaltantes despues de imputar (sin contar parent_event_id):")
restantes = df[cols_a_tratar].isna().sum()
print(restantes[restantes > 0] if restantes.sum() > 0 else "(ninguno)")
assert df[cols_a_tratar].isna().sum().sum() == 0, "Quedan faltantes sin tratar"
assert df['parent_event_id'].isna().sum() == df['seed_flag'].sum(), "parent_event_id nulo deberia ser exactamente 1 por cascada (la semilla)"
print("\nOK: no quedan faltantes salvo parent_event_id en las semillas (esperado).")


Faltantes antes de imputar (sin contar parent_event_id):
impressions                 625
comments                    936
dwell_time_sec             2005
content_half_life_min       666
sentiment_score            1344
device_fingerprint_code    1324
dtype: int64



Faltantes despues de imputar (sin contar parent_event_id):
(ninguno)

OK: no quedan faltantes salvo parent_event_id en las semillas (esperado).


## Paso 6 — Derivar campos nuevos

- **`nivel_audiencia`**: nano / micro / macro / mega, a partir de `actor_follower_count` (cortes definidos en 2.1 a partir de la distribución real por cuenta).
- **`hora_del_dia`**, **`dia_semana`**: extraídos de `forward_datetime`, para el análisis de no estacionariedad de 3.3.

In [8]:
bins = [0, 10_000, 100_000, 1_000_000, np.inf]
labels = ['nano', 'micro', 'macro', 'mega']
df['nivel_audiencia'] = pd.cut(df['actor_follower_count'], bins=bins, labels=labels, right=False)

df['hora_del_dia'] = df['forward_datetime'].dt.hour
df['dia_semana'] = df['forward_datetime'].dt.day_name()

print("Distribución de nivel_audiencia (eventos):")
print((df['nivel_audiencia'].value_counts(normalize=True) * 100).round(2))

assert df['nivel_audiencia'].isna().sum() == 0
assert df['hora_del_dia'].between(0, 23).all()
print("\nOK: campos derivados.")


Distribución de nivel_audiencia (eventos):
nivel_audiencia
nano     86.54
micro     6.76
macro     6.19
mega      0.50
Name: proportion, dtype: float64

OK: campos derivados.


## Paso 7 — Estandarizar unidades de tiempo

Todas las columnas temporales quedan en **minutos**, para que la tabla maestra de 3.4 tenga una U.M. consistente (`dwell_time_sec` era la única en segundos).

In [9]:
df['dwell_time_min'] = df['dwell_time_sec'] / 60
df = df.drop(columns=['dwell_time_sec'])

cols_tiempo = ['dwell_time_min', 'inter_event_time_min', 'content_half_life_min', 'moderation_latency_min']
print("Columnas temporales, todas en minutos:", cols_tiempo)
print(df[cols_tiempo].describe().loc[['mean', 'min', 'max']])


Columnas temporales, todas en minutos: ['dwell_time_min', 'inter_event_time_min', 'content_half_life_min', 'moderation_latency_min']
      dwell_time_min  inter_event_time_min  content_half_life_min  moderation_latency_min
mean        1.644744             64.387004             439.096219               55.717351
min         0.000000              0.000000              15.000000                0.000000
max       102.335000          57241.950000            4993.500000            26328.500000


## Resumen final y exportación

Dataset limpio, listo para el ajuste de distribuciones (3.3) y la tabla maestra de cuantificación (3.4).

In [10]:
print(f"Filas: {len(df):,} (de 66.153 originales)")
print(f"Cascadas: {df['cascade_id'].nunique():,} (de 5.000 originales)")
print(f"Columnas: {df.shape[1]} (se agregaron nivel_audiencia, hora_del_dia, dia_semana, dwell_time_min)")
print(f"Faltantes totales: {df.isna().sum().sum()} (todos en parent_event_id, uno por semilla — esperado)")
print(f"event_id duplicados: {df['event_id'].duplicated().sum()}")

df.to_csv(RUTA_LIMPIO, index=False)
print(f"\nGuardado en: {RUTA_LIMPIO}")


Filas: 66,021 (de 66.153 originales)
Cascadas: 5,000 (de 5.000 originales)
Columnas: 33 (se agregaron nivel_audiencia, hora_del_dia, dia_semana, dwell_time_min)
Faltantes totales: 5000 (todos en parent_event_id, uno por semilla — esperado)
event_id duplicados: 0



Guardado en: ../Datos/muestra_cascadas_REDBAIRES_limpio.csv
